In [0]:
# CAMADA GOLD - Data Marts e KPIs de negocio
# Origem: tabelas da camada Silver
# Projeto 1: Visao Comercial e Volume de Produtos
# Projeto 2: Satisfacao de Clientes e Qualidade de Parceiros

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

print("Banco de dados 'gold' criado com sucesso.")

Banco de dados 'gold' criado com sucesso.


In [0]:
# Camada Gold - fat_vendas_comercial
# Granularidade: Ano, Mes e Categoria de Produto
# Metricas: total_pedidos, qtd_itens_vendidos, receita_total_brl,
# receita_total_usd e ticket_medio_brl

from pyspark.sql.functions import (
    col, year, month, countDistinct, count,
    sum as spark_sum, round as spark_round
)

df_pedido_total = spark.table("silver.fat_pedido_total")
df_itens        = spark.table("silver.fat_itens_pedidos")
df_produtos     = spark.table("silver.dim_produtos")
df_traducao     = spark.table("silver.dim_categoria_produtos_traducao")

# Junta itens com produtos para obter categoria
df_itens_cat = df_itens.join(df_produtos, on="id_produto", how="left")

# Junta com traducao para obter nome da categoria em ingles
df_itens_cat = df_itens_cat.join(
    df_traducao,
    df_itens_cat["categoria_produto"] == df_traducao["nome_produto_pt"],
    how="left"
)

# Junta com pedido total para obter data e valores financeiros
df_itens_completo = df_itens_cat.join(
    df_pedido_total.select("id_pedido", "data_pedido", "valor_total_pago_brl", "valor_total_pago_usd"),
    on="id_pedido",
    how="left"
)

# Adiciona ano e mes
df_itens_completo = df_itens_completo \
    .withColumn("ano_venda", year(col("data_pedido"))) \
    .withColumn("mes_venda", month(col("data_pedido")))

# Agrega por ano, mes e categoria
df_vendas_comercial = df_itens_completo.groupBy(
    "ano_venda", "mes_venda", "nome_produto_en"
).agg(
    countDistinct("id_pedido").alias("total_pedidos"),
    count("id_item").alias("qtd_itens_vendidos"),
    spark_round(spark_sum("valor_total_pago_brl"), 2).alias("receita_total_brl"),
    spark_round(spark_sum("valor_total_pago_usd"), 2).alias("receita_total_usd"),
    spark_round(spark_sum("valor_total_pago_brl") / countDistinct("id_pedido"), 2).alias("ticket_medio_brl")
).withColumnRenamed("nome_produto_en", "categoria_produto")

(df_vendas_comercial.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fat_vendas_comercial"))

print(f"Tabela gold.fat_vendas_comercial criada com {df_vendas_comercial.count()} registros.")

Tabela gold.fat_vendas_comercial criada com 1274 registros.


In [0]:
# Rankings Comerciais - Projeto 1
# Top 5 produtos MAIS vendidos e Top 5 MENOS vendidos
# Exibidos via display() conforme solicitado na atividade

from pyspark.sql.functions import col, count

df_itens    = spark.table("silver.fat_itens_pedidos")
df_produtos = spark.table("silver.dim_produtos")
df_traducao = spark.table("silver.dim_categoria_produtos_traducao")

# Junta itens com produtos e traducao
df_ranking = df_itens \
    .join(df_produtos, on="id_produto", how="left") \
    .join(
        df_traducao,
        df_produtos["categoria_produto"] == df_traducao["nome_produto_pt"],
        how="left"
    ) \
    .groupBy("id_produto", "nome_produto", "nome_produto_en") \
    .agg(count("id_item").alias("quantidade_vendida")) \
    .withColumnRenamed("nome_produto_en", "categoria_produto")

print("--- Top 5 Produtos MAIS Vendidos ---")
df_mais_vendidos = df_ranking.orderBy(col("quantidade_vendida").desc()).limit(5)
display(df_mais_vendidos)

print("--- Top 5 Produtos MENOS Vendidos ---")
df_menos_vendidos = df_ranking.orderBy(col("quantidade_vendida").asc()).limit(5)
display(df_menos_vendidos)

--- Top 5 Produtos MAIS Vendidos ---


id_produto,nome_produto,categoria_produto,quantidade_vendida
aca2eb7d00ea1a7b8ebd4e68314663af,Estante de Livros Luxo,furniture_decor,527
99a4788cb24856965c36a24e339b6058,Cobertor Cinza,bed_bath_table,488
422879e10f46682990de24d770e7f83d,Cortador de Grama Branco,garden_tools,484
389d119b48cf3043d311335e499d9c6b,Kit de Ferramentas Ultra,garden_tools,392
368c6c730842d78016ad823897a372db,Kit de Ferramentas Master,garden_tools,388


--- Top 5 Produtos MENOS Vendidos ---


id_produto,nome_produto,categoria_produto,quantidade_vendida
cfbc837967ca671ee87f5243f2a1024d,Raquete de Tênis Pro,sports_leisure,1
d64e758afad411049a45e42c9a259241,Perfume,perfumery,1
d143bf43abb18593fa8ed20cc990ae84,Teclado Mecânico Plus,computers_accessories,1
8b41f2becf919f6177cca1ff15dfd311,Equipamento Utilitário,books_technical,1
40161099308e1f5f784d57e2b88b31b5,Creme Anti-idade,health_beauty,1


In [0]:
# Camada Gold - fat_avaliacoes_clientes
# Granularidade: Categoria do Produto, Nome do Vendedor e Estado
# Metricas: total_avaliacoes, avaliacao_media,
# total_avaliacoes_positivas, total_avaliacoes_negativas
# e percentual_satisfacao

from pyspark.sql.functions import (
    col, count, round as spark_round,
    avg, sum as spark_sum, when
)

df_avaliacoes = spark.table("silver.fat_avaliacoes_pedidos")
df_itens      = spark.table("silver.fat_itens_pedidos")
df_produtos   = spark.table("silver.dim_produtos")
df_vendedores = spark.table("silver.dim_vendedores")
df_traducao   = spark.table("silver.dim_categoria_produtos_traducao")

# Junta avaliacoes com itens para obter produto e vendedor
df = df_avaliacoes.join(
    df_itens.select("id_pedido", "id_produto", "id_vendedor"),
    on="id_pedido", how="left"
)

# Junta com produtos para obter categoria
df = df.join(df_produtos.select("id_produto", "categoria_produto"), on="id_produto", how="left")

# Junta com traducao
df = df.join(
    df_traducao,
    df["categoria_produto"] == df_traducao["nome_produto_pt"],
    how="left"
)

# Junta com vendedores para obter nome e estado
df = df.join(
    df_vendedores.select("id_vendedor", "nome_vendedor", "estado"),
    on="id_vendedor", how="left"
)

# Agrega as metricas
df_avaliacoes_clientes = df.groupBy(
    "nome_produto_en", "nome_vendedor", "estado"
).agg(
    count("id_avaliacao").alias("total_avaliacoes"),
    spark_round(avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    spark_sum(when(col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
    spark_sum(when(col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
).withColumn(
    "percentual_satisfacao",
    spark_round(col("total_avaliacoes_positivas") / col("total_avaliacoes") * 100, 2)
).withColumnRenamed("nome_produto_en", "categoria_produto")

(df_avaliacoes_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fat_avaliacoes_clientes"))

print(f"Tabela gold.fat_avaliacoes_clientes criada com {df_avaliacoes_clientes.count()} registros.")

Tabela gold.fat_avaliacoes_clientes criada com 6485 registros.


In [0]:
# Rankings de Qualidade - Projeto 2
# Criterio de ordenacao composto: nota (desc ou asc) e
# volume de avaliacoes (desc) como criterio de desempate
# Registros sem categoria ou vendedor sao excluidos do ranking

from pyspark.sql.functions import col

df_aval = spark.table("gold.fat_avaliacoes_clientes") \
    .filter(col("categoria_produto").isNotNull()) \
    .filter(col("nome_vendedor").isNotNull())

print("--- Produto MAIS Bem Avaliado ---")
df_produto_melhor = df_aval.orderBy(
    col("avaliacao_media").desc(),
    col("total_avaliacoes").desc()
).select("categoria_produto", "avaliacao_media", "total_avaliacoes").limit(1)
display(df_produto_melhor)

print("--- Produto MENOS Bem Avaliado ---")
df_produto_pior = df_aval.orderBy(
    col("avaliacao_media").asc(),
    col("total_avaliacoes").desc()
).select("categoria_produto", "avaliacao_media", "total_avaliacoes").limit(1)
display(df_produto_pior)

print("--- Vendedor MAIS Bem Avaliado ---")
df_vendedor_melhor = df_aval.orderBy(
    col("avaliacao_media").desc(),
    col("total_avaliacoes").desc()
).select("nome_vendedor", "estado", "avaliacao_media", "total_avaliacoes").limit(1)
display(df_vendedor_melhor)

print("--- Vendedor MENOS Bem Avaliado ---")
df_vendedor_pior = df_aval.orderBy(
    col("avaliacao_media").asc(),
    col("total_avaliacoes").desc()
).select("nome_vendedor", "estado", "avaliacao_media", "total_avaliacoes").limit(1)
display(df_vendedor_pior)

--- Produto MAIS Bem Avaliado ---


categoria_produto,avaliacao_media,total_avaliacoes
computers_accessories,5.0,17


--- Produto MENOS Bem Avaliado ---


categoria_produto,avaliacao_media,total_avaliacoes
art,1.0,7


--- Vendedor MAIS Bem Avaliado ---


nome_vendedor,estado,avaliacao_media,total_avaliacoes
Mariah Vargas,SP,5.0,17


--- Vendedor MENOS Bem Avaliado ---


nome_vendedor,estado,avaliacao_media,total_avaliacoes
Maria Clara Pastor,PR,1.0,7


In [0]:
# Otimizacao fisica das tabelas da camada Gold
# OPTIMIZE compacta arquivos e ZORDER organiza os dados
# pelas colunas de filtragem mais frequentes

tabelas_otimizar = [
    ("gold.fat_vendas_comercial",    "ano_venda, mes_venda"),
    ("gold.fat_avaliacoes_clientes", "categoria_produto"),
]

for tabela, colunas in tabelas_otimizar:
    spark.sql(f"OPTIMIZE {tabela} ZORDER BY ({colunas})")
    print(f"Tabela {tabela} otimizada com ZORDER por ({colunas}).")

print("Otimizacao da camada Gold concluida.")

Tabela gold.fat_vendas_comercial otimizada com ZORDER por (ano_venda, mes_venda).
Tabela gold.fat_avaliacoes_clientes otimizada com ZORDER por (categoria_produto).
Otimizacao da camada Gold concluida.
